# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
print("Dataset metadata summary:")
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All record sets, fields, and columns are referenced by their `@id` property, following Croissant best practices.

In [ ]:
# List all record sets and their IDs from the dataset
record_sets = dataset.metadata.recordSets()
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")

# Choose one record set (main data table) for demonstration
# In most cases, there is one main record set; if more than one, select appropriately
main_record_set = None
if record_sets:
    main_record_set = record_sets[0]

print("\nFields and columns in the selected Record Set:")
if main_record_set:
    for field in main_record_set.fields():
        print(f"  - Field @id: {field.id:40} | name: {field.name:30} | Data type: {field.dataType}")
        # Each field is usually linked to a column (column @id):
        if hasattr(field, 'column') and field.column:
            print(f"      Linked column @id: {field.column}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All references use their respective `@id`s.

In [ ]:
# Get all record set @ids
record_set_ids = [rs.id for rs in record_sets]

# We'll work with the main record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Explore columns in the main record set DataFrame
if main_record_set:
    main_rs_id = main_record_set.id
    if main_rs_id in dataframes:
        print(f"Columns for Record Set '@id': {main_rs_id}")
        print(dataframes[main_rs_id].columns.tolist())
        display(dataframes[main_rs_id].head())
    else:
        print(f"No records extracted for main record set {main_rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Remember: All fields referenced by their `@id` as per Croissant best practices.

In [ ]:
# Prepare for EDA: List fields and pick a numeric field
if main_record_set:
    # Find numeric fields (Float, Integer, Number)
    numeric_fields = [f for f in main_record_set.fields() if f.dataType in ["schema:Float", "schema:Integer", "schema:Number"]]

    if numeric_fields:
        numeric_field = numeric_fields[0]  # Pick first numeric field for demo
        print(f"Example numeric field: @id={numeric_field.id}, name={numeric_field.name}, type={numeric_field.dataType}")
        df = dataframes[main_record_set.id]

        # Ensure the field is numeric in the DataFrame
        df[numeric_field.id] = pd.to_numeric(df[numeric_field.id], errors='coerce')

        # Filter records where the field value is > threshold
        threshold = df[numeric_field.id].mean()
        filtered_df = df[df[numeric_field.id] > threshold].copy()
        print(f"Filtered records with {numeric_field.id} (mean={threshold:.2f}):")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field.id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field.id] - filtered_df[numeric_field.id].mean()) / filtered_df[numeric_field.id].std()
        print(f"Normalized values for '{numeric_field.id}':")
        display(filtered_df[[numeric_field.id, norm_col]].head())

        # Optionally, group by a categorical field (pick first non-numeric field)
        non_numeric_fields = [f for f in main_record_set.fields() if f.dataType not in ["schema:Float", "schema:Integer", "schema:Number"]]
        if non_numeric_fields:
            group_field = non_numeric_fields[0]  # First string/categorical
            if group_field.id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field.id)[numeric_field.id].mean().reset_index()
                print(f"Grouped average of '{numeric_field.id}' by '{group_field.id}':")
                display(grouped_df.head())
    else:
        print("No numeric field found in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set and numeric_fields:
    df = dataframes[main_record_set.id]
    num_id = numeric_fields[0].id
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[num_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {num_id}")
    plt.xlabel(num_id)
    plt.ylabel("Frequency")
    plt.show()

    # If there is a second numeric field, scatter plot
    if len(numeric_fields) > 1:
        num_id2 = numeric_fields[1].id
        plt.figure(figsize=(7, 5))
        sns.scatterplot(x=df[num_id], y=df[num_id2])
        plt.xlabel(num_id)
        plt.ylabel(num_id2)
        plt.title(f"Scatter plot: {num_id} vs {num_id2}")
        plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded and explored the dataset metadata using `mlcroissant`.
- Examined the available record sets and their fields using Croissant `@id` conventions.
- Loaded tabular data into Pandas DataFrames for further analysis.
- Performed basic EDA: filtering, normalization, and grouping by field `@id`s.
- Visualized key numeric fields and explored their distributions.

Next steps might include advanced statistical analysis or combining these data with external sources. For reference, always use the `@id` for reproducible and FAIR-compliant processing.